# Les 03 - Agentische Ontwerp Patronen

In deze les verkennen we drie fundamentele ontwerp patronen voor het bouwen van effectieve AI-agenten:

1. **Duidelijke Agent Instructies** — Het maken van nauwkeurige, rolbepalende prompts die het gedrag van de agent sturen  
2. **Gestructureerde Output met Pydantic Modellen** — Ervoor zorgen dat agenten voorspelbare, gevalideerde data retourneren  
3. **Agenten met Enkele Verantwoordelijkheid** — Het ontwerpen van gerichte agenten die elk één ding goed doen  

We passen elk patroon toe op een **reisbestemmingsaanbeveler** scenario, waarbij we stapsgewijs een systeem bouwen dat bestemmingen kan voorstellen, beschikbaarheid kan controleren en de logistiek kan afhandelen.


## Installatie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Patroon 1: Duidelijke Agentinstructies

Het meest impactvolle patroon is ook het eenvoudigste: het schrijven van duidelijke, gedetailleerde instructies voor uw agent.

Goede instructies definiëren:
- **Wie** de agent is (persoonlijkheid en toon)
- **Wat** hij moet doen (stapsgewijze verantwoordelijkheden)
- **Hoe** hij zich moet gedragen (beperkingen en stijl)

Hieronder creëren we een reisconcierge-agent met expliciete instructies die elke reactie die hij produceert vormen.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Patroon 2: Gestructureerde Uitvoer met Pydantic-modellen

Vrije tekst is nuttig voor conversatie, maar downstream-systemen hebben gestructureerde data nodig.
Door **Pydantic-modellen** te combineren met een **toolfunctie**, kunnen we:

- Een exact schema definiëren voor de output van de agent
- Reacties automatisch valideren
- Agentresultaten betrouwbaar integreren in applicatielogica

We introduceren ook een tool die bestemmingsgegevens retourneert zodat de agent zijn aanbevelingen baseert op echte data.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Patroon 3: Agents met Enkele Verantwoordelijkheid

Complexe taken profiteren van het verdelen van werk over meerdere gespecialiseerde agents, elk met een enkele verantwoordelijkheid:

- Een **Bestemmingsexpert** die kennis heeft over plaatsen en beschikbaarheid
- Een **Logistiek Planner** die vluchten, hotels en reisschema's regelt

Dit weerspiegelt het software engineering principe van *scheiding van verantwoordelijkheden* — elke agent is makkelijker te testen, onderhouden en zelfstandig te verbeteren.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Samenvatting

In deze les hebben we drie agentgerichte ontwerpprincipes toegepast op een reissuggestie-scenario:

| Patroon | Kernidee | Voordeel |
|---|---|---|
| **Duidelijke Instructies** | Definieer persona, verantwoordelijkheden en beperkingen vooraf | Consistent, merktrouw gedrag van de agent |
| **Gestructureerde Output** | Gebruik Pydantic-modellen als responsformaat | Gevalideerde, machineleesbare resultaten |
| **Enkele Verantwoordelijkheid** | Geef elke agent één gerichte taak | Makkelijker te testen, onderhouden en combineren |

Deze patronen combineren natuurlijk — je kunt duidelijke instructies combineren met gestructureerde output binnen een enkele-verantwoordelijkheidsagent om robuuste, productieklare systemen te bouwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
